# Topics of the 1914 Press

What was *L'Humanité* writing about in 1914, and how did the news turn that summer? Without a single label, BERTopic groups similar passages and describes each group by its most characteristic words.

1. Find the topics
2. Tune them
3. Follow them over time

The passages' *embeddings* are precomputed, with Qwen3-Embedding-0.6B, and truncated to 512 of its 1,024 dimensions: the model is trained so that its first dimensions are enough, and the truncation is already done. This notebook needs no GPU.

## Data

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/embeddings-512.npy
%pip install -q "bertopic<1"

In [ ]:
import numpy as np
import pandas as pd

passages = pd.read_parquet("passages.parquet")
embeddings = np.load("embeddings-512.npy").astype(np.float32)

# L'Humanité only (Le Figaro has only 24 issues), and a sample of 20,000 passages so
# that the computation takes a few minutes.
humanite = passages[passages.newspaper == "L'Humanité"].sample(20_000, random_state=0)
docs = humanite.text.tolist()
dates = humanite.date.tolist()
vectors = embeddings[humanite.index]
print(len(docs), vectors.shape)

## 1. Find the Topics

Create a `BERTopic` model that reduces the embeddings to 5 dimensions with UMAP (reproducibly), describes topics with a `CountVectorizer` without stop words (they are removed to **describe the topics**, a **lexical method**: c-TF-IDF counts each topic's words; never before the embeddings, which the model computes on the whole text), and only keeps topics of at least 50 passages. The corpus is French, so are its stop words.

Fit it with `fit_transform` (on the precomputed vectors), then display `get_topic_info()`: how many topics? **Share without a topic**: what share of the passages has no topic (topic $-1$)?

**Read**: go through the keywords of the 15 largest topics, and two representative passages (`get_representative_docs`) of three of them. The texts come from the OCR of the 1914 issues: garbled words, and classified ads, market prices and advertisements, which make up their own topics (the `noise` column of `humanite` gives each passage's share of unknown words). Which keywords say nothing about their topic? Write them down: exercise 2 adds them to the stop words.

**Useful documentation**

- [`BERTopic`](https://maartengr.github.io/BERTopic/api/bertopic.html)
- [`BERTopic.fit_transform`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.fit_transform)
- [`BERTopic.get_topic_info`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_topic_info)
- [`BERTopic.get_representative_docs`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_representative_docs)
- [UMAP's parameters](https://umap-learn.readthedocs.io/en/latest/parameters.html)
- [`CountVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from spacy.lang.fr.stop_words import STOP_WORDS
from umap import UMAP

STOP_WORDS = sorted(STOP_WORDS | {"qu", "mm", "mme"})

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0)`: a reduction to 5 dimensions, reproducible
- `CountVectorizer(stop_words=STOP_WORDS, min_df=10)`: BERTopic counts the words over one document per topic (all its passages put end to end), so `min_df=10` ignores the words found in fewer than 10 *topics*, not 10 passages
- `BERTopic(language="french", umap_model=..., vectorizer_model=..., min_topic_size=50)`: without `language="french"`, BERTopic, which gets precomputed vectors here (with no `embedding_model`), strips from the texts, before counting the words, every character other than an ASCII letter, a digit or a space (accents, apostrophes, punctuation): "grève" becomes "grve"
- `fit_transform(docs, vectors)`: the second argument avoids computing the embeddings again; returns each passage's topic, then probabilities
- `get_topic_info()`: one row per topic, topic −1 being the passages with no topic; `np.mean(np.array(topics) == -1)`: their share
- `get_topic(topic)`: a topic's keywords and their weights, of which `[word for word, _ in ...]` keeps the words; `get_representative_docs(topic)`: its representative passages

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
topic_model = BERTopic(
  language="french",
  umap_model=UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0
  ),
  vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
  min_topic_size=...,
)
topics, _ = topic_model.fit_transform(
  ..., ...
)  # the texts, then the precomputed vectors
info = topic_model.get_topic_info()
print(
  len(info) - 1, "topics,", f"{np.mean(np.array(topics) == -1):.0%} without a topic"
)
info.head(16)

for topic in info.Topic[1:4]:
  print(topic, [word for word, _ in topic_model.get_topic(topic)[:8]])
  for doc in topic_model.get_representative_docs(...)[:2]:
    print("   ", doc[:200])
```

### Solution

In [ ]:
topic_model = BERTopic(
  language="french",
  umap_model=UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0
  ),
  vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
  min_topic_size=50,
)
topics, _ = topic_model.fit_transform(docs, vectors)
info = topic_model.get_topic_info()
print(
  len(info) - 1, "topics,", f"{np.mean(np.array(topics) == -1):.0%} without a topic"
)
info.head(16)

In [ ]:
for topic in info.Topic[1:4]:
  print(topic, [word for word, _ in topic_model.get_topic(topic)[:8]])
  for doc in topic_model.get_representative_docs(topic)[:2]:
    print("   ", doc[:200])

# The figures of these comments (sizes, numbers and ids of the topics) change from one
# installation to another, with the versions of numpy and numba: look for what they
# illustrate. Here, the largest topic (3,228 passages) mixes reports from the front and
# serialized novels: its first representative passage is a dialogue from a novel, the
# second an account of the battle of the Yser. On your side, look among the large topics
# (widen `infos.Topic[1:4]` if need be) for the one whose representative passages are
# about two things. Keywords like "bien", "été", "rien", "oui", "fut", "faire" or
# "contre" (well, been, nothing, yes, was, do, against) say nothing about the topic.
# Classified ads ("fr, rue, paris, 50, 75") and advertisements have topics of their own.

## 2. Tune the Topics

- Rerun with `min_topic_size=20`, then `100`: how many topics, what share without a topic?
- **Stability**: rerun the first model with another seed for UMAP (`random_state=1`). Do its large topics come back? For each of the five largest topics of the first model, find the new topic that receives most of its passages (`pd.crosstab`), and what share.
- Add the keywords that say nothing, written down in exercise 1, to the stop words, and recompute the first model's keywords (`update_topics`): without training again, the topics stay the same, only their description changes.
- Find two topics about the same thing and merge them (`merge_topics`).
- **Share without a topic**: finally, reassign the passages without a topic to their nearest topic (`reduce_outliers`), then update the descriptions (`update_topics`). The merge comes first: once topics have been assigned by hand, like the reassignment's, reducing the number of topics can fail, and BERTopic reminds you with a warning. Read five reassigned passages: do they belong where they are?

**Useful documentation**

- [Outlier reduction](https://maartengr.github.io/BERTopic/getting_started/outlier_reduction/outlier_reduction.html)
- [`BERTopic.update_topics`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.update_topics)
- [`BERTopic.merge_topics`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.merge_topics)
- [`pandas.crosstab`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- rerunning with another size: in a loop over `(20, 100)`, a new `BERTopic` with the same configuration, with `min_topic_size=size` (without overwriting `topic_model`, the first model)
- another seed: a new `BERTopic` with the same configuration, with `UMAP(..., random_state=1)` and `min_topic_size=50`
- `pd.crosstab(np.array(topics), np.array(seed_1_topics))`: the number of passages of each first topic (in rows) in each new topic (in columns); `crossed.loc[topic]`: a topic's row; `.drop(-1, errors="ignore").idxmax()`: the new topic that receives the most of them, leaving topic −1 aside
- `sorted(set(STOP_WORDS) | {...})`: the stop words, plus yours; `topic_model.update_topics(docs, vectorizer_model=CountVectorizer(stop_words=PRESS_STOP_WORDS, min_df=10))`: without `topics=`, the topics stay those of the training, only the keywords are recomputed
- `topic_model.merge_topics(docs, [a, b])`: `a` and `b` are two topics to merge into one, picked by reading the keywords of `get_topic_info()`. Topic numbers change after a merge: `merge_topics` renumbers every topic and returns nothing (`None`), so take `topics = topic_model.topics_` again right after it
- `topic_model.reduce_outliers(docs, topics, strategy="embeddings", embeddings=vectors)` returns the passages' new topics
- `topic_model.update_topics(docs, topics=new_topics, vectorizer_model=...)`: recomputes the keywords after the reassignment, with your stop words
- `pd.Series(new_topics)[np.array(topics) == -1].sample(5, random_state=0)`: five passages that had no topic, with their new topic; `docs[i]`: the text of passage `i`

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
for size in (20, 100):
  model = BERTopic(
    language="french",
    umap_model=UMAP(
      n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0
    ),
    vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
    min_topic_size=...,
  )
  found, _ = model.fit_transform(docs, vectors)
  print(
    f"min_topic_size={size}: {len(set(found)) - 1} topics, {np.mean(np.array(found) == -1):.0%} without a topic"
  )

seed_1_model = BERTopic(
  language="french",
  umap_model=UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=...
  ),
  vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
  min_topic_size=50,
)
seed_1_topics, _ = seed_1_model.fit_transform(docs, vectors)
crossed = pd.crosstab(np.array(topics), np.array(seed_1_topics))
for topic in info.Topic[1:6]:  # the five largest first topics
  row = crossed.loc[topic]
  closest = row.drop(-1, errors="ignore").idxmax()
  words = [word for word, _ in seed_1_model.get_topic(closest)[:5]]
  print(topic, f"{row[closest] / row.sum():.0%} in topic {closest}", words)

PRESS_STOP_WORDS = sorted(set(STOP_WORDS) | {...})  # your keywords that say nothing
topic_model.update_topics(
  docs, vectorizer_model=CountVectorizer(stop_words=..., min_df=10)
)

topic_model.merge_topics(docs, [..., ...])  # two topics about the same thing
topics = topic_model.topics_  # the numbers have changed

new_topics = topic_model.reduce_outliers(docs, topics, strategy=..., embeddings=vectors)
topic_model.update_topics(docs, topics=new_topics, vectorizer_model=...)
print(f"{np.mean(np.array(new_topics) == -1):.0%} without a topic after reassignment")
reassigned = pd.Series(new_topics)[np.array(topics) == -1].sample(5, random_state=0)
for i, topic in reassigned.items():
  print(topic, [word for word, _ in topic_model.get_topic(topic)[:5]])
  print("   ", docs[i][:200])
topics = new_topics
```

### Solution

In [ ]:
for size in (20, 100):
  model = BERTopic(
    language="french",
    umap_model=UMAP(
      n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0
    ),
    vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
    min_topic_size=size,
  )
  found, _ = model.fit_transform(docs, vectors)
  print(
    f"min_topic_size={size}: {len(set(found)) - 1} topics, {np.mean(np.array(found) == -1):.0%} without a topic"
  )

# Stability: the first model, with another seed for UMAP.
seed_1_model = BERTopic(
  language="french",
  umap_model=UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=1
  ),
  vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
  min_topic_size=50,
)
seed_1_topics, _ = seed_1_model.fit_transform(docs, vectors)
print(
  f"seed 1: {len(set(seed_1_topics)) - 1} topics, {np.mean(np.array(seed_1_topics) == -1):.0%} without a topic"
)
crossed = pd.crosstab(np.array(topics), np.array(seed_1_topics))
for topic in info.Topic[1:6]:  # the five largest first topics
  row = crossed.loc[topic]
  closest = row.drop(-1, errors="ignore").idxmax()
  words = [word for word, _ in seed_1_model.get_topic(closest)[:5]]
  print(topic, f"{row[closest] / row.sum():.0%} in topic {closest}", words)

# Here, with seed 1: 44 topics, 38% without a topic (other versions of numpy and numba
# give other figures). What to look for: the clear-cut topics come back nearly whole
# (here local news, meetings, sport: 94 to 99% of their passages in a single topic; the
# strikes at 58%), and a topic that scatters between new topics and no topic was a mix.
# Here, the largest comes apart: 32% of its passages go to a topic of novels ("bien,
# yeux, rien, tête": well, eyes, nothing, head), 35% to no topic. If no large topic
# comes apart on your side, they hold with this seed: stability is judged over two or
# three seeds (2, 3).

In [ ]:
# The keywords that say nothing, found in exercise 1.
PRESS_STOP_WORDS = sorted(
  set(STOP_WORDS) | {"été", "bien", "rien", "oui", "non", "fut", "faire", "contre"}
)
topic_model.update_topics(
  docs, vectorizer_model=CountVectorizer(stop_words=PRESS_STOP_WORDS, min_df=10)
)
for topic in info.Topic[1:4]:
  print(topic, [word for word, _ in topic_model.get_topic(topic)[:8]])

# Here, without "bien", "été", "rien" and "oui", the mixed topic's keywords are
# "allemands, yeux, hommes, troupes, ennemi, nuit" (Germans, eyes, men, troops, enemy,
# night): war and novels, still mixed. The topics themselves haven't moved: only their
# description changed.

In [ ]:
# Here, two topics are about the legislative elections of April-May 1914: the candidates
# on one side, the abbreviated result tables on the other. Their numbers change with the
# libraries' versions: we find them through their shared keyword, "voix" (votes).
elections = [
  topic
  for topic in topic_model.get_topics()
  if "voix" in [word for word, _ in topic_model.get_topic(topic)]
]
for topic in elections:
  print(topic, [word for word, _ in topic_model.get_topic(topic)[:8]])

topic_model.merge_topics(docs, elections)
topics = topic_model.topics_  # the merge renumbered the topics

# The reassignment comes last: BERTopic's warning is a reminder of it.
new_topics = topic_model.reduce_outliers(
  docs, topics, strategy="embeddings", embeddings=vectors
)
topic_model.update_topics(
  docs,
  topics=new_topics,
  vectorizer_model=CountVectorizer(stop_words=PRESS_STOP_WORDS, min_df=10),
)
print(f"{np.mean(np.array(new_topics) == -1):.0%} without a topic after reassignment")
reassigned = pd.Series(new_topics)[np.array(topics) == -1].sample(5, random_state=0)
for i, topic in reassigned.items():
  print(topic, [word for word, _ in topic_model.get_topic(topic)[:5]])
  print("   ", docs[i][:200])
topics = new_topics

# Of these five passages, the one about the Senate is where it belongs; the others are
# to be read in full: two, one of them an extract from a novel, join the large mixed
# topic of exercise 1. Reassigning puts every passage somewhere, not necessarily in the
# right place.

## 3. Follow the Topics over Time

`topic_model.topics_over_time` counts each topic's passages per week; `topic_model.visualize_topics_over_time` plots them. Show the topics about the war (*guerre*), Jaurès and the Caillaux trial: write a function `topic_with(word)` that returns the topic with the largest share of passages containing `word`.

Before plotting: does the newspaper publish as many passages every month? Count the sample's, month by month. Then plot, rather than each topic's number of passages, their **share** of the week's passages. What happens between July and August 1914?

**Useful documentation**

- [Topics over time](https://maartengr.github.io/BERTopic/getting_started/topicsovertime/topicsovertime.html)
- [`BERTopic.topics_over_time`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.topics_over_time)
- [`BERTopic.visualize_topics_over_time`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.visualize_topics_over_time)
- [`Series.idxmax`](https://pandas.pydata.org/docs/reference/api/pandas.Series.idxmax.html)
- [`SeriesGroupBy.transform`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.SeriesGroupBy.transform.html)
- [`DataFrame.assign`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `topic_model.topics_over_time(docs, dates, nr_bins=52)`: 52 bins, a week each over a year; a table with one row per topic and per week, with the topic (`Topic`), its number of passages (`Frequency`) and the week (`Timestamp`)
- `topic_with(word)`: among the passages containing the word (`word in doc.lower()`), the number of passages of each topic (`pd.Series(topics)[mask].value_counts()`), divided by that topic's total size (`pd.Series(topics).value_counts()`), without topic −1 (`.drop(-1, errors="ignore")`)
- `Series.idxmax()`: the label of the largest value, here the topic's number; try `"guerre"`, `"jaurès"` and `"caillaux"`
- `humanite.groupby(humanite.date.dt.month).size()`: the number of passages of each month
- `over_time.groupby("Timestamp").Frequency.transform("sum")`: for each row, the number of passages of its week; `over_time.assign(Frequency=over_time.Frequency / totals)`: the same table, with shares
- `topic_model.visualize_topics_over_time(shares, topics=chosen)`: `topics=` is the list of topics to plot. Its `normalize_frequency=True` option doesn't settle the question: it brings each curve to its own scale, not to the newspaper's volume

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
over_time = topic_model.topics_over_time(docs, dates, nr_bins=...)


def topic_with(word: str) -> int:
  mentions = pd.Series(topics)[[word in doc.lower() for doc in docs]].value_counts()
  share = (mentions / pd.Series(topics).value_counts()).drop(-1, errors="ignore")
  return int(share.idxmax())


chosen = [topic_with(word) for word in ("guerre", "jaurès", ...)]
print(chosen)
print(humanite.groupby(humanite.date.dt.month).size())  # passages per month
totals = over_time.groupby("Timestamp").Frequency.transform(...)
shares = over_time.assign(Frequency=over_time.Frequency / totals)
topic_model.visualize_topics_over_time(..., topics=...)
```

### Solution

In [ ]:
over_time = topic_model.topics_over_time(docs, dates, nr_bins=52)


def topic_with(word: str) -> int:
  mentions = pd.Series(topics)[[word in doc.lower() for doc in docs]].value_counts()
  share = (mentions / pd.Series(topics).value_counts()).drop(-1, errors="ignore")
  return int(share.idxmax())


chosen = [topic_with(word) for word in ("guerre", "jaurès", "caillaux")]
print(chosen)
print(humanite.groupby(humanite.date.dt.month).size())  # passages per month
totals = over_time.groupby("Timestamp").Frequency.transform("sum")
shares = over_time.assign(Frequency=over_time.Frequency / totals)
topic_model.visualize_topics_over_time(shares, topics=chosen)

# The sample holds 2,400 passages in July, 872 in August: the newspaper still comes out
# every day, but its issues get shorter. In numbers, the war topic goes from 77 passages
# in July to 120 in August; in share, from 3% to 14%. In March, Mme Caillaux kills
# Gaston Calmette, the editor of Le Figaro; her trial fills July (10% of the passages in
# the weeks of the 16th and the 23rd). Jaurès is assassinated on July 31: 17% of the
# passages of the week of the 30th.